# Notebook 08 — FTL vs Carting Decision Framework

ML-backed route-type selection:
- LightGBM + Isotonic calibration
- ROC curve, Calibration curve, Decision boundary analysis
- Scenario analysis: time-cost trade-off quantification

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, calibration_curve
from src.models.ftl_carting import train, predict_route_type, build_training_set, FEATURES
print('FTL/Carting module loaded')

In [ ]:
df = pd.read_parquet('data/processed/features.parquet')
# Add graph features if available
try:
    from src.models.baseline import add_graph_features
    df = add_graph_features(df)
except Exception as e:
    print(f'Graph features unavailable: {e}')
if 'is_intercity' not in df.columns:
    df['is_intercity'] = 0

print(f'Dataset: {df.shape}')
if 'route_type' in df.columns:
    print(df['route_type'].value_counts())

In [ ]:
# Train model
model = train(df)
print('Model trained and saved.')

In [ ]:
# Scenario analysis: how does FTL probability vary with distance and delay?
dist_vals = np.linspace(50, 2000, 40)
delay_vals = [1.05, 1.20, 1.40, 1.60]

fig = go.Figure()
for delay in delay_vals:
    scenario = pd.DataFrame({
        'osrm_distance': dist_vals,
        'time_of_day_enc': 2,
        'corridor_mean_delay': delay,
        'corridor_volume': 200,
        'src_betweenness': 0.15,
        'src_pagerank': 0.03,
        'dwell_time_proxy': 15,
        'is_intercity': (dist_vals > 400).astype(int),
        'route_type': 'FTL',
    })
    result = predict_route_type(scenario)
    fig.add_trace(go.Scatter(x=dist_vals, y=result['ftl_probability'],
                             name=f'Delay ratio = {delay}', mode='lines'))

fig.add_hline(y=0.55, line_dash='dash', line_color='yellow', annotation_text='Decision threshold (0.55)')
fig.update_layout(title='P(FTL) vs Distance for Different Corridor Delay Profiles',
                  xaxis_title='Distance (km)', yaxis_title='P(FTL recommended)',
                  template='plotly_dark')
fig.show()
fig.write_html('reports/08_ftl_decision_boundary.html')

In [ ]:
# Time-cost trade-off table
sample_trips = pd.DataFrame({
    'osrm_distance': [100, 300, 600, 1000, 1400],
    'time_of_day_enc': [1, 2, 3, 0, 1],
    'corridor_mean_delay': [1.05, 1.15, 1.35, 1.50, 1.65],
    'corridor_volume': [500, 300, 150, 80, 40],
    'src_betweenness': [0.05, 0.10, 0.20, 0.30, 0.35],
    'src_pagerank': [0.01, 0.02, 0.04, 0.06, 0.07],
    'dwell_time_proxy': [5, 10, 20, 30, 40],
    'is_intercity': [0, 0, 1, 1, 1],
    'route_type': ['Carting', 'Carting', 'FTL', 'FTL', 'FTL'],
})
result = predict_route_type(sample_trips)
print(result[['osrm_distance','route_type_recommendation','ftl_probability',
              'expected_time_saving_min','cost_premium_pct','decision_reason']].to_string())